# Multi-ETF Municipal Bond Screener — Master

This repo notebook is synchronized to the uploaded **multistate** master version. It uses the shared `src/muni_data.py` engine so Jupyter and Streamlit do not drift.

Master-specific behavior: multiple states can be selected, and the `No State Individual Income Tax` filter uses Alaska, Florida, Nevada, New Hampshire, South Dakota, Tennessee, Texas, and Wyoming. Washington is intentionally excluded.


In [ ]:
%pip install -q -U pandas numpy requests beautifulsoup4


In [ ]:
from pathlib import Path
import sys
import pandas as pd

repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.muni_data import load_all_ishares_munis, screen_munis

NO_INDIVIDUAL_INCOME_TAX_STATES = {
    'Alaska', 'Florida', 'Nevada', 'New Hampshire',
    'South Dakota', 'Tennessee', 'Texas', 'Wyoming'
}


In [ ]:
df, source_rows, etf_status, as_of = load_all_ishares_munis()
print(f'Unique municipal CUSIPs: {len(df):,}')
print(f'Raw ETF holding rows: {len(source_rows):,}')
print(f'Latest source date: {as_of}')
display(etf_status.sort_values(['Status', 'Ticker']).reset_index(drop=True))


## Master multistate screen

Edit the values below. `states=[]` means all states. Exact CUSIP lookup overrides the other filters, matching the uploaded notebook.


In [ ]:
CUSIP = ''
STATES = []  # Example: ['Indiana', 'Ohio', 'Michigan']
NO_STATE_INDIVIDUAL_INCOME_TAX = False
PURCHASE_FACE = 5000
PRICE_MIN = None
PRICE_MAX = None
COUPON_MIN = None
COUPON_MAX = None
YTW_MIN = None
YTW_MAX = None
MATURITY_FROM = None
MATURITY_TO = None
INVESTMENT_GRADE_ONLY = False
AMT_EXEMPT_ONLY = False
NON_CALLABLE_ONLY = False
NEW_ISSUE_ONLY = False
NEW_ISSUE_DAYS = 60
SORT_BY = 'YTW: High → Low'


In [ ]:
# Match the uploaded master: CUSIP overrides all other filters.
screen_df = df
if not CUSIP.strip() and NO_STATE_INDIVIDUAL_INCOME_TAX:
    screen_df = screen_df[screen_df['State'].isin(NO_INDIVIDUAL_INCOME_TAX_STATES)].copy()

results = screen_munis(
    df=screen_df,
    cusip=CUSIP,
    states=STATES or None,
    purchase_face=PURCHASE_FACE,
    price_min=PRICE_MIN,
    price_max=PRICE_MAX,
    coupon_min=COUPON_MIN,
    coupon_max=COUPON_MAX,
    ytw_min=YTW_MIN,
    ytw_max=YTW_MAX,
    maturity_from=MATURITY_FROM,
    maturity_to=MATURITY_TO,
    investment_grade_only=INVESTMENT_GRADE_ONLY,
    amt_exempt_only=AMT_EXEMPT_ONLY,
    non_callable_only=NON_CALLABLE_ONLY,
    new_issue_only=NEW_ISSUE_ONLY,
    new_issue_days=NEW_ISSUE_DAYS,
    as_of=as_of,
    sort_by=SORT_BY,
)

print(f'Matches: {len(results):,} / {len(df):,}')
display_cols = [
    'CUSIP', 'Name', 'State', 'Price', 'Coupon (%)', 'YTM (%)',
    'Yield to Worst (%)', 'Yield to Call (%)', 'Maturity', 'Rating',
    'Investment Grade', 'AMT Exempt', 'Source ETFs', 'Source Count',
    'Est. Principal Cost ($)', 'Annual Coupon Income ($)', 'NonCallableProxy'
]
display(results[[c for c in display_cols if c in results.columns]].head(250))
